In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    mutual_info_score
)
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier  






In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hopesb/student-depression-dataset",
    output_dir="./student_depression_dataset")

print("Path to dataset files:", path)




Path to dataset files: ./student_depression_dataset


In [4]:
df= pd.read_csv(path+"/Student Depression Dataset.csv")
df.head()

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,M.Tech,Yes,1.0,1.0,No,0


In [5]:
df.columns=df.columns.str.lower().str.replace(" ", "_")
for col in df.columns:
    if (df[col].dtype=="str"):
        df[col]= df[col].str.lower().str.replace(" ","_")



In [6]:
del df["id"]
del df["profession"]
del df["work_pressure"]
del df["job_satisfaction"]
del df["city"]

In [7]:
df=df.dropna().reset_index(drop=True)
categorical_columns=["gender","sleep_duration","dietary_habits","degree","have_you_ever_had_suicidal_thoughts_?","family_history_of_mental_illness"]
numerical_columns=["age","academic_pressure","cgpa","study_satisfaction","work/study_hours","financial_stress"]
df.head()

,gender,age,academic_pressure,cgpa,study_satisfaction,sleep_duration,dietary_habits,degree,have_you_ever_had_suicidal_thoughts_?,work/study_hours,financial_stress,family_history_of_mental_illness,depression
0,male,33.0,5.0,8.97,2.0,5-6_hours,healthy,b.pharm,yes,3.0,1.0,no,1
1,female,24.0,2.0,5.90,5.0,5-6_hours,moderate,bsc,no,3.0,2.0,yes,0
2,male,31.0,3.0,7.03,5.0,less_than_5_hours,healthy,ba,no,9.0,1.0,yes,0
3,female,28.0,3.0,5.59,2.0,7-8_hours,moderate,bca,yes,4.0,5.0,yes,1
4,female,25.0,4.0,8.13,3.0,5-6_hours,moderate,m.tech,yes,1.0,1.0,no,0


In [8]:
from sklearn.model_selection import train_test_split

df_full_train, df_test=train_test_split(df, test_size=0.2, random_state=5234)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=5234)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [9]:
y_train=df_train["depression"]
y_val=df_val["depression"]
y_test=df_test["depression"]

In [10]:
del df_train["depression"]
del df_val["depression"]
del df_test["depression"]

In [11]:
def train_random_forest(df_train, y_train, max_depth,min_samples_leaf):
    dv=DictVectorizer(sparse=False)
    X_dict=df_train.to_dict(orient="records")
    dv.fit(X_dict)
    X_train=dv.transform(X_dict)
    model=RandomForestClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=123234)
    model.fit(X_train, y_train)
    return dv, model

def predict_random_forest(df_val, dv:DictVectorizer, model:RandomForestClassifier):
    X_train=dv.transform(df_val.to_dict(orient="records"))
    return model.predict_proba(X_train)[:,1]




In [12]:
def train_test_random_forest(max_depth, min_samples_leaf):
    kfold= KFold(n_splits=5,shuffle=True, random_state=2344)
    scores=[]
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train=df_full_train.iloc[train_idx]
        df_val=df_full_train.iloc[val_idx]
        y_train = df_train.depression.values
        y_val = df_val.depression.values
        df_train=df_train.drop(columns=["depression"])
        df_val=df_val.drop(columns=["depression"])
        dv, model = train_random_forest(df_train, y_train, max_depth=max_depth,min_samples_leaf=min_samples_leaf)
        y_pred = predict_random_forest(df_val, dv, model)
        auc = roc_auc_score(y_val, y_pred)
        scores.append(auc)
    scores=np.array(scores)
    return scores.mean()

In [13]:
for md in [2,5,10,15,20,30,40]:
    auc_score=train_test_random_forest(md,4)
    print(f"max_depth={md} roc_auc_score={auc_score}")
    

max_depth=2 roc_auc_score=0.8937573605960427
max_depth=5 roc_auc_score=0.9011958662193166
max_depth=10 roc_auc_score=0.9121567127830168
max_depth=15 roc_auc_score=0.9138535039187786
max_depth=20 roc_auc_score=0.9139648483917068
max_depth=30 roc_auc_score=0.9136519418805971
max_depth=40 roc_auc_score=0.9136540123922868


In [14]:
for min_samples_leaf in [3,4,5,6,7,8]:
    auc_score=train_test_random_forest(20,min_samples_leaf)
    print(f"min_samples_leaf={min_samples_leaf} roc_auc_score={auc_score}")

min_samples_leaf=3 roc_auc_score=0.9139202422039533
min_samples_leaf=4 roc_auc_score=0.9139648483917068
min_samples_leaf=5 roc_auc_score=0.9141247339903185
min_samples_leaf=6 roc_auc_score=0.9140458674784411
min_samples_leaf=7 roc_auc_score=0.9135769051159744
min_samples_leaf=8 roc_auc_score=0.9131458465405924


In [17]:
final_model=RandomForestClassifier(max_depth=20, min_samples_leaf=5, random_state=235234)
dv=DictVectorizer(sparse=False)
X_train=dv.fit_transform(df_full_train.drop(columns=["depression"]).to_dict(orient="records"))
y_train=df_full_train.depression
final_model.fit(X_train, y_train)
X_test=dv.transform(df_test.to_dict(orient="records"))
y_pred=final_model.predict_proba(X_test)[:,1]


In [18]:

y_pred_class = (y_pred >= 0.5).astype(int)


accuracy = accuracy_score(y_test, y_pred_class)
recall = recall_score(y_test, y_pred_class)
f1 = f1_score(y_test, y_pred_class)
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_class)


print("Accuracy:  ", round(accuracy, 4))
print("Recall:    ", round(recall, 4))
print("F1 Score:  ", round(f1, 4))
print("AUC:       ", round(auc, 4))
print("Precision: ", round(precision, 4))

Accuracy:   0.8462
Recall:     0.8893
F1 Score:   0.8709
AUC:        0.9201
Precision:  0.8531
